# 📊 Extracción de Tablas de PDF a Excel
### Cómo convertir tablas de un PDF institucional en hojas de Excel automáticamente

**Caso real:** Informe "Plan Anual de Vacantes 2025" (CNSC)

---
*Este notebook está diseñado como una presentación: cada sección explica un paso del proceso, seguido del código que lo implementa.*


## 1️⃣ El problema

Un PDF institucional puede tener **decenas de tablas** con:

- Encabezados que ocupan varias líneas (*"TOTAL CARGOS / EN / PROVISIONALIDAD"*)
- Celdas fusionadas verticalmente (un valor que aplica a 2 filas)
- Gráficos de barras que visualmente parecen tablas

**Hacerlo a mano** → horas de copiar y pegar, con errores.
**Hacerlo con código** → segundos, de forma repetible.

> 💡 **Resumen simple:** vamos a enseñarle a Python a "leer" las tablas del PDF igual que las leería una persona, y a guardarlas en Excel, una hoja por tabla.


## 2️⃣ Herramientas que usamos

| Librería | Para qué sirve |
|---|---|
| `pdfplumber` | Leer el PDF y detectar tablas a partir de sus bordes dibujados |
| `pandas` | Organizar los datos extraídos en tablas (DataFrames) |
| `openpyxl` | Escribir el archivo Excel final, con formato |

**Resumen simple:** son 3 cajas de herramientas. Una abre el PDF, otra ordena los datos, otra escribe el Excel.


In [ ]:
# Instalar las librerías necesarias (solo se hace una vez por sesión de Colab)
!pip install pdfplumber openpyxl -q

from google.colab import files
import pdfplumber, pandas as pd, re
from openpyxl.styles import Font, PatternFill, Alignment

print("✅ Librerías listas")


## 3️⃣ Subir el archivo PDF

Colab necesita que subamos el archivo desde nuestro computador antes de poder leerlo.

**Resumen simple:** es como adjuntar un archivo en un correo, pero dentro del notebook.


In [ ]:
# Abre un selector de archivos. Sube tu PDF cuando se te pida.
uploaded = files.upload()

# Tomamos automáticamente el nombre del archivo subido
pdf_path = list(uploaded.keys())[0]
print(f"📄 Archivo cargado: {pdf_path}")


## 4️⃣ El descubrimiento clave: ¿por qué fallaba la primera versión?

Al revisar el PDF de cerca, descubrimos algo importante:

El documento usa **bordes de tabla muy delgados** (menos de 1 punto de grosor) que, sin querer, `pdfplumber` interpretaba como **columnas separadas**. Esto producía:

- Tablas "fantasma" con columnas vacías
- Encabezados partidos en varias tablas falsas
- Párrafos de texto confundidos con tablas

**Resumen simple:** Python veía "líneas" donde en realidad solo había decoración del diseño del PDF, no datos reales.

---

### 🔬 Detalle técnico (para quien quiera profundizar)

`pdfplumber.extract_tables()` puede detectar tablas de dos formas:
- `vertical_strategy="lines"` / `horizontal_strategy="lines"`: usa **líneas y rectángulos realmente dibujados** en el PDF.
- Por defecto, sin ajustar tolerancias, **cada borde diminuto cuenta como una columna nueva**.

La solución es indicarle a `pdfplumber` que **fusione bordes que estén muy cerca entre sí** y que **ignore bordes demasiado cortos** (ruido visual):

```python
TABLE_SETTINGS = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "snap_tolerance": 5,    # funde bordes a ≤5pt de distancia
    "join_tolerance": 5,    # une segmentos de línea cercanos
    "edge_min_length": 10,  # ignora bordes más cortos que 10pt
}
```


In [ ]:
# Configuración que usaremos para TODAS las páginas del PDF
TABLE_SETTINGS = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "snap_tolerance": 5,
    "join_tolerance": 5,
    "edge_min_length": 10,
}

print("⚙️ Configuración de extracción lista")


## 5️⃣ Filtrar los gráficos de barras

Los gráficos de barras del informe (ilustraciones) generan una cuadrícula que `pdfplumber`
puede confundir con una tabla. La diferencia es que estas "tablas falsas" tienen
**casi todas sus celdas vacías**.

**Resumen simple:** si una "tabla" está casi vacía, probablemente no es una tabla — es un gráfico.


In [ ]:
def es_tabla_real(tabla, min_llenado=0.35):
    """
    Decide si una tabla detectada es real o es ruido (ej. un gráfico).
    Cuenta cuántas celdas tienen contenido vs. cuántas están vacías.
    """
    total = sum(len(fila) for fila in tabla)
    llenas = sum(1 for fila in tabla for c in fila if c and str(c).strip())
    return total > 0 and (llenas / total) >= min_llenado

print("🧹 Filtro de tablas reales listo")


## 6️⃣ Reconocer títulos y limpiar encabezados

Algunas tablas tienen un título que ocupa toda la fila de arriba
(ej. *"ALCALDÍAS CIUDADES CAPITALES"*), y los encabezados de columna a veces
vienen partidos en varias líneas dentro de la misma celda
(ej. *"TOTAL CARGOS\nEN\nPROVISIONALIDAD"*).

**Resumen simple:**
- Si una fila tiene un solo texto que ocupa todo el ancho → es un título, no una columna.
- Si una celda tiene saltos de línea → los unimos en una sola frase.


In [ ]:
def es_fila_titulo(fila):
    """¿Esta fila es un título que abarca toda la tabla?"""
    no_vacias = [c for c in fila if c and str(c).strip()]
    return len(no_vacias) == 1 and len(fila) > 1


def limpiar_encabezado(valor):
    """Convierte 'TOTAL CARGOS\nEN\nPROVISIONALIDAD' en una sola línea de texto."""
    if valor is None:
        return ""
    texto = str(valor).replace("\n", " ")
    return re.sub(r'\s+', ' ', texto).strip()

print("🏷️ Funciones de títulos y encabezados listas")


## 7️⃣ Resolver celdas fusionadas verticalmente

Algunas tablas tienen un valor (ej. el total general) que visualmente ocupa
**dos filas** en el PDF, pero al extraerlo solo aparece en la primera fila
y la segunda queda vacía.

**Resumen simple:** "estiramos" ese valor hacia abajo para que aparezca en ambas filas.


In [ ]:
def fill_down(df):
    """
    Propaga un valor hacia abajo cuando una celda quedó vacía
    por una fusión vertical en el PDF original.
    """
    return df.replace("", pd.NA).ffill(axis=0).fillna("")

print("⬇️ Función fill-down lista")


## 8️⃣ Armar la tabla final (DataFrame)

Con todas las piezas anteriores, ahora construimos una función que toma
una tabla "cruda" (como la extrae `pdfplumber`) y devuelve:

1. El **título**, si existe
2. Una **tabla ordenada** (DataFrame de pandas) con encabezados limpios

**Resumen simple:** esta función es el "ensamblaje final" que junta todo lo anterior.


In [ ]:
def tabla_a_dataframe(tabla):
    titulo = ""
    filas = [list(f) for f in tabla]

    # Si la primera fila es un título, lo separamos del resto
    if es_fila_titulo(filas[0]):
        titulo = limpiar_encabezado([c for c in filas[0] if c][0])
        filas = filas[1:]

    # Limpiamos saltos de línea de todas las celdas
    filas = [[limpiar_encabezado(c) for c in fila] for fila in filas]

    # La primera fila restante son los encabezados de columna
    encabezados = filas[0]
    datos = filas[1:]

    # Si hay encabezados vacíos o repetidos, les damos un nombre único
    vistos, nombres = {}, []
    for j, h in enumerate(encabezados):
        h = h if h else f"Columna_{j+1}"
        if h in vistos:
            vistos[h] += 1
            nombres.append(f"{h}_{vistos[h]}")
        else:
            vistos[h] = 0
            nombres.append(h)

    df = pd.DataFrame(datos, columns=nombres)
    df = fill_down(df)
    return titulo, df

print("🧩 Función de ensamblaje lista")


## 9️⃣ Recorrer todo el PDF

Ahora sí: recorremos **cada página** del PDF, extraemos las tablas con la
configuración correcta, descartamos los gráficos, y guardamos cada tabla real
en una lista junto con su página y su título.

**Resumen simple:** Python pasa página por página, "mirando" si hay una tabla real,
y si la hay, la guarda en una lista para después escribirla en Excel.


In [ ]:
todas = []

with pdfplumber.open(pdf_path) as pdf:
    for num_pag, pag in enumerate(pdf.pages, 1):
        tablas_crudas = pag.extract_tables(table_settings=TABLE_SETTINGS)

        for num_t, tabla in enumerate(tablas_crudas, 1):
            if not es_tabla_real(tabla):
                continue  # Es un gráfico, no una tabla real -> se ignora

            titulo, df = tabla_a_dataframe(tabla)
            todas.append({"pagina": num_pag, "numero": num_t,
                          "titulo": titulo, "datos": df})

            etiqueta = f"[{titulo}] " if titulo else ""
            print(f"✅ Pág {num_pag} · Tabla {num_t} {etiqueta}-> {df.shape[0]} filas x {df.shape[1]} columnas")

print(f"\n📊 Total de tablas reales extraídas: {len(todas)}")


## 🔟 Exportar todo a un archivo Excel

Por último, escribimos un archivo `.xlsx` donde **cada tabla queda en su propia hoja**,
nombrada según la página y el número de tabla (ej. `Pag21_T1`).

Si la tabla tenía un título, lo escribimos en la primera fila con fondo azul, para que se vea igual que en el PDF original.

**Resumen simple:** así obtenemos un Excel ordenado, fácil de revisar y de usar para análisis posterior.


In [ ]:
nombre_excel = "tablas_extraidas.xlsx"
usados = {}

with pd.ExcelWriter(nombre_excel, engine="openpyxl") as writer:
    for t in todas:
        base = f"Pag{t['pagina']}_T{t['numero']}"
        hoja = base[:31]  # Excel limita el nombre de hoja a 31 caracteres

        # Evitar nombres de hoja duplicados
        usados[hoja] = usados.get(hoja, 0) + 1
        if usados[hoja] > 1:
            hoja = f"{hoja[:28]}_{usados[hoja]}"

        fila_inicio = 2 if t["titulo"] else 1
        t["datos"].to_excel(writer, sheet_name=hoja, startrow=fila_inicio - 1, index=False)

        if t["titulo"]:
            ws = writer.sheets[hoja]
            celda = ws.cell(row=1, column=1, value=t["titulo"])
            celda.font = Font(bold=True, color="FFFFFF")
            celda.fill = PatternFill("solid", fgColor="4472C4")
            celda.alignment = Alignment(horizontal="center")

print(f"💾 Archivo generado: {nombre_excel}")

# Descargar el archivo a tu computador
files.download(nombre_excel)
print("⬇️ Descarga iniciada")


## ✅ Resumen del proceso completo

| Paso | Qué hace | Por qué es necesario |
|---|---|---|
| 1. Configurar `pdfplumber` | Fusiona micro-bordes del PDF | Evita columnas/tablas fantasma |
| 2. Filtrar tablas reales | Descarta cuadrículas vacías | Evita confundir gráficos con tablas |
| 3. Detectar títulos | Separa la fila de título | Evita que el título se mezcle con los datos |
| 4. Limpiar encabezados | Une saltos de línea | Encabezados legibles en una sola celda |
| 5. Fill-down | Propaga celdas fusionadas | Ningún dato queda vacío por error |
| 6. Exportar a Excel | Una hoja por tabla | Resultado ordenado, fácil de revisar |

---

### 🔬 Nota técnica final
El ajuste más importante de todo el proceso fue `snap_tolerance` y `edge_min_length`
en `TABLE_SETTINGS`. Sin este ajuste, `pdfplumber` toma literalmente cada borde dibujado
en el PDF como una división de columna — y muchos PDFs institucionales usan bordes
decorativos muy finos que no representan columnas de datos reales.

**Esto es generalizable:** si extraes tablas de otro PDF y ves columnas vacías o tablas
fragmentadas, el primer lugar a revisar es esta configuración.
